# Hierarchical Clustering

## Import data

This data comprises of the top three principal components obtained during PCA with a radial basis function kernel, which together contain ~95% of the total variance of the original dataset (as seen in the cumulative variance barplot from the PCA section).

In [1]:
import pandas as pd

df = pd.read_csv('../../data/kpca3d.csv')
df.head()

,PC1,PC2,PC3,All Time Rank,All Time Rank Bin,Track,Artist
0,0.150251,0.630721,-0.248971,1,1,MILLION DOLLAR BABY,Tommy Richman
1,-0.089829,-0.025684,-0.041813,2,1,Not Like Us,Kendrick Lamar
2,-0.021122,0.281381,-0.197222,3,1,i like the way you kiss me,Artemas
3,0.600836,0.611817,-0.103087,4,1,Flowers,Miley Cyrus
4,0.010005,0.400341,-0.205579,6,1,Lovin On Me,Jack Harlow


## Normalize data

In [2]:
from sklearn.preprocessing import StandardScaler

def normalize_data(df, quantitative_cols):
    """Use standard scaler to normalize quantitative data columns."""
    scaler = StandardScaler()
    return scaler.fit_transform(df[quantitative_cols])

In [3]:
df_scaled = normalize_data(df, ['PC1', 'PC2', 'PC3'])
df_scaled = pd.DataFrame(df_scaled)
df_scaled = df_scaled.rename(columns={0: 'PC1', 1: 'PC2', 2: 'PC3'})
df_scaled.describe()

,PC1,PC2,PC3
count,3.311000e+03,3.311000e+03,3.311000e+03
mean,-3.433610e-17,-2.575208e-17,8.584026e-18
std,1.000151e+00,1.000151e+00,1.000151e+00
min,-8.435623e-01,-1.562760e+00,-2.589789e+00
25%,-6.970511e-01,-5.067461e-01,-5.929945e-01
50%,-4.307679e-01,-9.299233e-02,4.344051e-02
75%,3.206849e-01,9.725160e-02,5.452515e-01
max,3.540588e+00,4.897467e+00,6.095500e+00


## Separate label and principal components

In [4]:
# extract label
label = df[['All Time Rank Bin']]

# extract three principal components
df_pca = df_scaled[['PC1', 'PC2', 'PC3']].values
pd.DataFrame(df_pca).head()

,0,1,2
0,0.562439,3.470437,-1.954463
1,-0.336259,-0.141324,-0.328238
2,-0.079065,1.548252,-1.548226
3,2.249132,3.366418,-0.809250
4,0.037453,2.202810,-1.613826


## Import modules

For hierarchical clustering, we use scipy's linkage, dendrogram, and fcluster:
1. linkage: https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
2. dendrogram: https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html
3. fcluster: https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.fcluster.html

For plotting, we use Plotly's 3D scatterplot, seaborn, and Pyplot:
1. 3D: https://plotly.com/python/3d-scatter-plots/
2. seaborn: https://seaborn.pydata.org/
3. Pyplot: https://matplotlib.org/stable/tutorials/pyplot.html

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import numpy as np

## Sample because algorithm is computationally intensive

In [72]:
n = 1500
sample_indices = np.random.choice(df_pca.shape[0], n, replace=False)
df_pca_sampled = df_pca[sample_indices]

## Visualize dendrogram of a set of sampled data points

In [ ]:
def apply_hierarchical_clustering(X, method='average', distance_threshold=0.5):
    """Perform hierarchical clustering with average linkage method (distances measured from cluster average)."""
    
    # linkage matrix
    linkage_matrix = linkage(X, method=method)
    
    # dendrogram
    plt.figure(figsize=(20, 10))
    dendrogram(linkage_matrix, color_threshold=distance_threshold)
    plt.axhline(y=distance_threshold, color='mediumseagreen', linestyle=':', label=f'Threshold = {distance_threshold}', lw=3)
    plt.title(f'Dendrogram ({method} linkage)', fontsize=18)
    plt.ylabel('Distance')
    plt.xlabel('Song')
    plt.xticks([])
    plt.legend(loc='upper left', fontsize=18)
    plt.show()
    
    return linkage_matrix

distance_threshold = 0.5

linkage_matrix = apply_hierarchical_clustering(df_pca_sampled, distance_threshold=distance_threshold)

## Visualize clusters formed by hierarchical clustering

In [ ]:
def visualize_clusters_3d_hierarchical(X, linkage_matrix, distance_threshold, colors):
    """Create 3D scatterplots to visualize hierarchical clusters."""
    
    # extract clusters based on cutoff distance
    labels = fcluster(linkage_matrix, t=distance_threshold, criterion='distance')
    cluster_colors = [f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})" for r, g, b in colors]

    # data points
    fig_3d = go.Figure(data=go.Scatter3d(
        x=X[:, 0], y=X[:, 1], z=X[:, 2],
        mode='markers',
        marker=dict(size=5, color=[cluster_colors[i] for i in labels], opacity=0.8),
        name='Data Points'
    ))

    title = f'3D Hierarchical Clustering (Distance Threshold = {distance_threshold})'
    fig_3d.update_layout(
        title=title,
        scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'),
        margin=dict(l=0, r=0, b=0, t=40),
        scene_camera=dict(
            eye=dict(x=1.3, y=-1.7, z=1.3), 
            center=dict(x=0, y=0, z=0), 
            up=dict(x=0, y=0, z=1)
        ),
        autosize=False, 
        width=1000, 
        height=800
    )

    # save Plotly's HTML files to display on a website
    fig_3d.write_html(f"./hierarchical-plots/{title.replace(' ', '')}.html")
    fig_3d.show()

# set up colormap
colors = sns.color_palette('Set2')
    
visualize_clusters_3d_hierarchical(df_pca_sampled, linkage_matrix, distance_threshold, colors)